In [15]:
import cv2
import torch
import logging
from datetime import datetime

In [16]:
logging.basicConfig(
    filename='smoking_detections.log',
    level=logging.INFO,
    format='%(asctime)s - %(message)s',
    datefmt='%Y-%m-%d %H:%M:%S'
)

In [17]:
# Load the pre-trained model
model = torch.hub.load('ultralytics/yolov5', 'custom', path='training/epochs_25/weights.pt')

Using cache found in C:\Users\admin/.cache\torch\hub\ultralytics_yolov5_master
YOLOv5  2025-4-30 Python-3.10.16 torch-2.1.0+cu118 CUDA:0 (NVIDIA GeForce GTX 1650, 4096MiB)

Fusing layers... 
Model summary: 157 layers, 7012822 parameters, 0 gradients, 15.8 GFLOPs
Adding AutoShape... 


In [18]:
# Define the window name
window_name = 'Smoking Detection'

# Open the webcam
cap = cv2.VideoCapture(0)

while True:
    ret, frame = cap.read()
    if not ret:
        break

    # Perform object detection
    results = model(frame)

    # Render the results on the frame
    annotated_frame = results.render()[0]

    # Display the frame
    cv2.imshow(window_name, annotated_frame)

    # Extract detection results
    detections = results.pandas().xyxy[0]

    # Iterate through detections
    for _, row in detections.iterrows():
        class_name = row['name']
        confidence = row['confidence']
        if class_name.lower() in ['smoke', 'smoking', 'cigarette']:
            logging.info(f"Detected {class_name} with confidence {confidence:.2f}")

    # Exit on pressing 'q' or closing the window
    if cv2.getWindowProperty(window_name, cv2.WND_PROP_VISIBLE) < 1:
        break
    if cv2.waitKey(1) & 0xFF == ord('q'):
        break


# Release resources
cap.release()
cv2.destroyAllWindows()